## load files from USGS databased download


In [ ]:
import pandas as pd
from pathlib import Path
from functools import reduce
# --- Shared / portable path ---
base_path = Path("data")
# --- Local path (for my own machine) ---
# base_path = Path('/home/st929/project/Shuang_Project')
# Load datasets
alk = pd.read_csv(base_path / 'alk_discharge_flux_year_daily.csv') 
ca  = pd.read_csv(base_path / 'ca_discharge_flux_year_daily.csv')   
temp = pd.read_csv(base_path / 'temp_daily.csv')                    
pH  = pd.read_csv(base_path / 'ph_discharge_flux_year_daily.csv')   
tds = pd.read_csv(base_path / 'tds_discharge_flux_year_daily.csv')  
mg  = pd.read_csv(base_path / 'mg_discharge_flux_year_daily.csv')  

In [ ]:
# Step 1: Take ca as the "base" with metadata + ca values
base = ca[['site_no', 'stream_date', 'station_nm', 'dec_lon_va', 'dec_lat_va',
           'MonitoringLocationTypeName', 'StateCode', 'CountyCode', 'hucCd',
           'stream_month', 'stream_year', 'stream_season',
           'value', 'discharge_l_d']].rename(
               columns={'value': 'ca', 'discharge_l_d': 'ca_discharge'}
           )

# Step 2: Extract and rename values + discharge from others
mg_ = mg[['site_no', 'stream_date', 'value', 'discharge_l_d']].rename(
    columns={'value': 'mg', 'discharge_l_d': 'mg_discharge'}
)
pH_ = pH[['site_no', 'stream_date', 'value', 'discharge_l_d']].rename(
    columns={'value': 'pH', 'discharge_l_d': 'pH_discharge'}
)
alk_ = alk[['site_no', 'stream_date', 'value', 'discharge_l_d']].rename(
    columns={'value': 'alk', 'discharge_l_d': 'alk_discharge'}
)
tds_ = tds[['site_no', 'stream_date', 'value', 'discharge_l_d']].rename(
    columns={'value': 'tds', 'discharge_l_d': 'tds_discharge'}
)

# temp doesn't have discharge
temp_ = temp[['site_no', 'stream_date', 'value']].rename(columns={'value': 'temp'})

# Step 3: Merge all together
dfs = [base, mg_, pH_, alk_, tds_, temp_]
merged = reduce(lambda left, right: pd.merge(left, right, on=['site_no', 'stream_date'], how='inner'), dfs)

# Step 4: Since all discharge columns are the same, keep only one
merged = merged.drop(columns=['mg_discharge', 'pH_discharge', 'alk_discharge', 'tds_discharge'])
merged = merged.rename(columns={'ca_discharge': 'discharge'})

# Step 5: Reorder so discharge is right after stream_date
col_order = ['site_no', 'stream_date', 'discharge',
             'station_nm', 'dec_lon_va', 'dec_lat_va',
             'MonitoringLocationTypeName', 'StateCode', 'CountyCode', 'hucCd',
             'stream_month', 'stream_year', 'stream_season',
             'ca', 'mg', 'pH', 'alk', 'tds', 'temp']

merged = merged[col_order]

# Step 6: Preview
print(merged.head())


## calculation of discharge quantiles

In [13]:
import pandas as pd
import numpy as np

# ---------- Constants ----------
CFS_TO_L_PER_DAY = 28.316846592 * 86400.0  # 1 cfs = ~2.4466e6 L/day

# ---------- 1) Build quantile table Q1–Q99 ----------
def build_station_quantiles_full(daily_discharge, min_rows=1000):
    dd = daily_discharge.copy()
    dd = dd[dd["unit"].str.lower().eq("cfs")]
    dd["value"] = pd.to_numeric(dd["value"], errors="coerce")
    dd = dd.dropna(subset=["value"])

    # Keep only stations with enough daily rows
    valid_sites = dd.groupby("site_no")["value"].count()
    valid_sites = valid_sites[valid_sites >= min_rows].index
    dd = dd[dd["site_no"].isin(valid_sites)]

    # Compute integer quantiles Q1–Q99 (avoid float rounding duplicates)
    quantiles = np.linspace(0.01, 0.99, 99)
    q = dd.groupby("site_no")["value"].quantile(quantiles).unstack().reset_index()

    # Rename columns safely to Qxx_Ld
    rename_map = {qval: f"Q{int(round(qval*100))}_Ld" for qval in quantiles}
    q = q.rename(columns=rename_map)

    # Convert to L/day
    for col in q.columns:
        if col.endswith("_Ld"):
            q[col] = q[col] * CFS_TO_L_PER_DAY
    return q

q_tbl = build_station_quantiles_full(daily_discharge, min_rows=1000)

## Assign a discharge quantile number (Q1-Q100) to each measurement 

In [ ]:

def assign_discharge_quantile(merged, quantiles_df, min_measurements=1):
    df = merged.copy()
    df["discharge_l_d"] = pd.to_numeric(df["discharge"], errors="coerce")

    # Keep only sites that also exist in quantiles_df
    df = df[df["site_no"].isin(quantiles_df["site_no"])]

    # Keep only sites with at least `min_measurements` chemistry rows
    valid_sites = df.groupby("site_no").size()
    valid_sites = valid_sites[valid_sites >= min_measurements].index
    df = df[df["site_no"].isin(valid_sites)]

    # Merge thresholds
    df = df.merge(quantiles_df, on="site_no", how="left")
    q_cols = [c for c in quantiles_df.columns if c.startswith("Q")]

    # Find closest quantile for each row
    def find_quantile(row):
        d = row["discharge_l_d"]
        if pd.isna(d):
            return np.nan
        vals = row[q_cols].values.astype(float)
        if len(vals) == 0 or np.all(pd.isna(vals)):
            return np.nan
        diffs = np.abs(vals - d)
        idx = diffs.argmin()
        return q_cols[idx].replace("_Ld", "")  # return e.g. "Q30"

    df["flow_quantile"] = df.apply(find_quantile, axis=1)

    # Drop thresholds to return clean merged + quantile
    keep_cols = merged.columns.tolist() + ["flow_quantile"]
    return df[keep_cols]

merged_filtered = merged[merged["site_no"].isin(q_tbl["site_no"])]
merged_with_q = assign_discharge_quantile(merged_filtered, q_tbl, min_measurements=1)

# Save
merged_with_q.to_csv("merged_with_flow_quantile_min1.csv", index=False)
print("Saved merged_with_flow_quantile.csv", merged_with_q.shape)


## calculate omega from parameters

In [ ]:
## run python code /home/st929/project/Shuang_Project/calculate_daily_omega.py
## run in cluster.

import PyCO2SYS as pyco2
import pandas as pd
import numpy as np
from pathlib import Path

# --- Paths ---
# Local path (for your own machine)
# input_path = Path("/home/st929/Shuang_Project_backup_from_projectfolder/merged_with_flow_quantile_min1.csv")
# output_path = Path("/home/st929/Shuang_Project_backup_from_projectfolder/merged_with_q_with_carbonate_params.csv")

# Shared / portable path
data_dir = Path("data")
input_path = data_dir / "merged_with_flow_quantile_min1.csv"
output_path = data_dir / "merged_with_q_with_carbonate_params.csv"

# Check data directory
if not data_dir.exists():
    raise FileNotFoundError("⚠️ 'data/' folder not found.")

# --- Load data ---
merged_with_q = pd.read_csv(input_path, index_col=False)

opt_pH_scale = 4  # NBS scale for freshwater

# --- Prepare result lists ---
omega_calcite_vals   = []
omega_aragonite_vals = []
k_calcite_vals       = []
k_aragonite_vals     = []
pco2_vals            = []
co3_vals             = []
dic_vals             = []
ph25_vals            = []
pco225_vals          = []
omega_calcite25_vals   = []
omega_aragonite25_vals = []

total_rows = len(merged_with_q)

# --- Loop over rows ---
for i, (idx, row) in enumerate(merged_with_q.iterrows(), start=1):
    try:
        alk_uM = row["alk"] * 1e6 if pd.notna(row["alk"]) else np.nan
        ca_uM  = row["ca"]  * 1e6 if pd.notna(row["ca"])  else np.nan
        temp_C = row["temp"]
        salinity = row["tds"] / 1000.0 if pd.notna(row["tds"]) else np.nan  # mg/L → g/L ≈ PSU

        # Convert [H+] to pH
        pH_val = -np.log10(row["pH"]) if (pd.notna(row["pH"]) and row["pH"] > 0) else np.nan

        if any(np.isnan(x) for x in [alk_uM, ca_uM, salinity, temp_C, pH_val]):
            omega_calcite_vals.append(np.nan)
            omega_aragonite_vals.append(np.nan)
            k_calcite_vals.append(np.nan)
            k_aragonite_vals.append(np.nan)
            pco2_vals.append(np.nan)
            co3_vals.append(np.nan)
            dic_vals.append(np.nan)
            ph25_vals.append(np.nan)
            pco225_vals.append(np.nan)
            omega_calcite25_vals.append(np.nan)
            omega_aragonite25_vals.append(np.nan)
            continue

        # --- Step 1: In-situ calculation (TA + pH) ---
        results = pyco2.sys(
            par1=alk_uM,
            par2=pH_val,
            par1_type=1,
            par2_type=3,
            salinity=salinity,
            temperature=temp_C,
            pressure=0,
            total_calcium=ca_uM,
            opt_pH_scale=opt_pH_scale,
            opt_k_carbonic=15
        )

        omega_calcite_vals.append(results["saturation_calcite"])
        omega_aragonite_vals.append(results["saturation_aragonite"])
        k_calcite_vals.append(float(results["k_calcite"]))
        k_aragonite_vals.append(float(results["k_aragonite"]))
        pco2_vals.append(results["pCO2"])
        co3_vals.append(results["CO3"])
        dic_vals.append(results["dic"])

        dic = results["dic"]

        # --- Step 2: Normalize to 25°C (TA + DIC) ---
        results25 = pyco2.sys(
            par1=alk_uM,
            par2=dic,
            par1_type=1,
            par2_type=2,
            salinity=salinity,
            temperature=25,
            pressure=0,
            total_calcium=ca_uM,
            opt_pH_scale=opt_pH_scale,
            opt_k_carbonic=15
        )

        ph25_vals.append(results25["pH"])
        pco225_vals.append(results25["pCO2"])
        omega_calcite25_vals.append(results25["saturation_calcite"])
        omega_aragonite25_vals.append(results25["saturation_aragonite"])

    except Exception as e:
        print(f"❌ Error at row {i}, site {row.get('site_no','NA')} date {row.get('stream_date','NA')}: {e}")

        omega_calcite_vals.append(np.nan)
        omega_aragonite_vals.append(np.nan)
        k_calcite_vals.append(np.nan)
        k_aragonite_vals.append(np.nan)
        pco2_vals.append(np.nan)
        co3_vals.append(np.nan)
        dic_vals.append(np.nan)
        ph25_vals.append(np.nan)
        pco225_vals.append(np.nan)
        omega_calcite25_vals.append(np.nan)
        omega_aragonite25_vals.append(np.nan)

    # Progress update
    if i % 50 == 0 or i == total_rows:
        print(f"Processed {i}/{total_rows} rows ({i/total_rows:.1%})")

# --- Assign results ---
merged_with_q["omega_calcite"]   = omega_calcite_vals
merged_with_q["omega_aragonite"] = omega_aragonite_vals
merged_with_q["Ksp_calcite"]     = k_calcite_vals
merged_with_q["Ksp_aragonite"]   = k_aragonite_vals
merged_with_q["pCO2"]            = pco2_vals
merged_with_q["CO3"]             = co3_vals
merged_with_q["DIC"]             = dic_vals
merged_with_q["pH_25C"]          = ph25_vals
merged_with_q["pCO2_25C"]        = pco225_vals
merged_with_q["omega_calcite_25C"]   = omega_calcite25_vals
merged_with_q["omega_aragonite_25C"] = omega_aragonite25_vals

# --- Save output ---
merged_with_q.to_csv(output_path, index=False)
print(f"✅ Finished — saved with carbonate parameters: {merged_with_q.shape}, file: {output_path}")

In [2]:
## load file again

In [ ]:
import pandas as pd

merged_with_q_with_omega = pd.read_csv(
    "/home/st929/Shuang_Project_backup_from_projectfolder/merged_with_q_with_carbonate_params.csv",
    index_col=False
)

In [ ]:
import pandas as pd

# Suppose your sodium dataset is called df_na
# and your omega dataset is called merged_with_q_with_omega
file_path = '/home/st929/project/Shuang_Project/na_discharge_flux_year_daily.csv'
na = pd.read_csv(file_path)

# First, pivot sodium so you get one column for 'Sodium' values
df_na = na.rename(columns={"value": "na"})  # rename 'value' to 'na' for clarity
# Merge on site_no and stream_date
merged_with_q_with_omega_na = pd.merge(
    merged_with_q_with_omega,
    df_na[["site_no", "stream_date", "na"]],  # keep only needed columns
    on=["site_no", "stream_date"],
    how="left"   # left join keeps all omega rows, adds na where available
)


## build and assign temperature quantile number (T1-T100) to each measurement 

In [ ]:
import pandas as pd
import numpy as np
file_path = '/home/st929/project/Shuang_Project/temp_daily.csv'
temp = pd.read_csv(file_path)
# ---------- 1) Build temperature quantile table Q1–Q99 ----------
def build_temp_quantiles(temp_df, min_rows=100):
    dd = temp_df.copy()
    dd["value"] = pd.to_numeric(dd["value"], errors="coerce")
    dd = dd.dropna(subset=["value"])

    # Keep only stations with enough rows
    valid_sites = dd.groupby("site_no")["value"].count()
    valid_sites = valid_sites[valid_sites >= min_rows].index
    dd = dd[dd["site_no"].isin(valid_sites)]

    # Compute integer quantiles Q1–Q99
    quantiles = np.linspace(0.01, 0.99, 99)
    q = dd.groupby("site_no")["value"].quantile(quantiles).unstack().reset_index()

    # Rename columns e.g., 0.01 → Q1_temp
    rename_map = {qval: f"T{int(round(qval*100))}_temp" for qval in quantiles}
    q = q.rename(columns=rename_map)

    return q


# ---------- 2) Assign temperature quantiles ----------
def assign_temp_quantile(merged, quantiles_df, min_measurements=1):
    df = merged.copy()
    df["temp"] = pd.to_numeric(df["temp"], errors="coerce")

    # Keep only sites that also exist in quantiles_df
    df = df[df["site_no"].isin(quantiles_df["site_no"])]

    # Keep only sites with at least `min_measurements` rows
    valid_sites = df.groupby("site_no").size()
    valid_sites = valid_sites[valid_sites >= min_measurements].index
    df = df[df["site_no"].isin(valid_sites)]

    # Merge thresholds
    df = df.merge(quantiles_df, on="site_no", how="left")
    q_cols = [c for c in quantiles_df.columns if c.startswith("T")]

    # Find closest quantile for each row
    def find_quantile(row):
        t = row["temp"]
        if pd.isna(t):
            return np.nan
        vals = row[q_cols].values.astype(float)
        if len(vals) == 0 or np.all(pd.isna(vals)):
            return np.nan
        diffs = np.abs(vals - t)
        idx = diffs.argmin()
        return q_cols[idx].replace("_temp", "")  # return e.g. "Q30"

    df["temp_quantile"] = df.apply(find_quantile, axis=1)

    # Drop thresholds to return clean merged + quantile
    keep_cols = merged.columns.tolist() + ["temp_quantile"]
    return df[keep_cols]


# ---------- Usage ----------
# temp = pd.read_csv("temp.csv")   # your file with water temp time series
# merged = pd.read_csv("merged_with_q_with_omega_na.csv")

q_tbl_temp = build_temp_quantiles(temp, min_rows=100)
merged_with_temp_q = assign_temp_quantile(merged_with_q_with_omega_na, q_tbl_temp, min_measurements=1)

# Save
merged_with_temp_q.to_csv("/home/st929/Shuang_Project_backup_from_projectfolder/merged_with_q_t_omega_na.csv", index=False)
print("Saved merged_with_temp_quantile.csv", merged_with_temp_q.shape)


### add sodium data to the database (didn't end up using in this study)

In [3]:
import pandas as pd

merged_with_q_with_omega = pd.read_csv(
    "/home/st929/Shuang_Project_backup_from_projectfolder/merged_with_q_with_carbonate_params.csv",
    index_col=False
)

# Suppose your sodium dataset is called df_na
# and your omega dataset is called merged_with_q_with_omega
file_path = '/home/st929/project/Shuang_Project/na_discharge_flux_year_daily.csv'
na = pd.read_csv(file_path)

# First, pivot sodium so you get one column for 'Sodium' values
df_na = na.rename(columns={"value": "na"})  # rename 'value' to 'na' for clarity
# Merge on site_no and stream_date
merged_with_q_with_omega_na = pd.merge(
    merged_with_q_with_omega,
    df_na[["site_no", "stream_date", "na"]],  # keep only needed columns
    on=["site_no", "stream_date"],
    how="left"   # left join keeps all omega rows, adds na where available
)


## create necessary files for WRTDS

In [3]:
import pandas as pd
from pathlib import Path

# --- Input files ---
input_files = {
    "alk": "/home/st929/project/Shuang_Project/alk_discharge_flux_year_daily.csv",
    "ca":  "/home/st929/project/Shuang_Project/ca_discharge_flux_year_daily.csv",
    "na":  "/home/st929/project/Shuang_Project/na_discharge_flux_year_daily.csv",
    "mg":  "/home/st929/project/Shuang_Project/mg_discharge_flux_year_daily.csv",
    "ph":  "/home/st929/project/Shuang_Project/ph_discharge_flux_year_daily.csv",
    "tds":  "/home/st929/project/Shuang_Project/tds_discharge_flux_year_daily.csv",
}

# --- Output base directory ---
base_out = Path("/home/st929/project/OMEGA_Drought_paper/sample_WRTDS")
base_out.mkdir(exist_ok=True)


# --- Function to detect censored values ---
def parse_censored(val):
    if isinstance(val, str) and val.strip().startswith("<"):
        try:
            lim = float(val.strip().replace("<", ""))
        except ValueError:
            lim = pd.NA
        return "<", lim
    else:
        return "", float(val)

# --- Process each analyte file ---
for analyte, fpath in input_files.items():
    out_dir = base_out / f"{analyte}_Sample"
    out_dir.mkdir(exist_ok=True)

    print(f"Processing {analyte} → {out_dir}")

    # Load data
    df = pd.read_csv(fpath, sep=",", parse_dates=["stream_date"])

    # Apply censor parser
    parsed = df["value"].apply(parse_censored)
    df["Remark"] = [p[0] for p in parsed]
    df["Conc"]   = [p[1] for p in parsed]

    # Keep only WRTDS-qualified sites
    df = df[df["site_no"].isin(sites_to_keep)]

    # Group by site and save
    for site, site_df in df.groupby("site_no"):
        site_out = site_df[["stream_date", "Remark", "Conc"]].copy()
        site_out.columns = ["Date", "Remark", "Conc"]

        # Format date
        site_out["Date"] = site_out["Date"].dt.strftime("%Y-%m-%d")

        site_out.to_csv(out_dir / f"{site}_Sample.csv", index=False)


Processing alk → /home/st929/project/OMEGA_Drought_paper/sample_WRTDS/alk_Sample
Processing ca → /home/st929/project/OMEGA_Drought_paper/sample_WRTDS/ca_Sample
Processing na → /home/st929/project/OMEGA_Drought_paper/sample_WRTDS/na_Sample
Processing mg → /home/st929/project/OMEGA_Drought_paper/sample_WRTDS/mg_Sample
Processing ph → /home/st929/project/OMEGA_Drought_paper/sample_WRTDS/ph_Sample
Processing tds → /home/st929/project/OMEGA_Drought_paper/sample_WRTDS/tds_Sample


In [ ]:
import pandas as pd
from pathlib import Path

# Input analyte files
input_files = {
    "alk": "/home/st929/project/Shuang_Project/alk_discharge_flux_year_daily.csv",
    "ca":  "/home/st929/project/Shuang_Project/ca_discharge_flux_year_daily.csv",
    "na":  "/home/st929/project/Shuang_Project/na_discharge_flux_year_daily.csv",
    "mg":  "/home/st929/project/Shuang_Project/mg_discharge_flux_year_daily.csv",
    "ph":  "/home/st929/project/Shuang_Project/ph_discharge_flux_year_daily.csv",
    "tds": "/home/st929/project/Shuang_Project/tds_discharge_flux_year_daily.csv",
}

# Output dir
info_out = Path("/home/st929/project/OMEGA_Drought_paper/info_WRTDS")
info_out.mkdir(exist_ok=True)

# Map to abbreviations
constit_map = {
    "Alkalinity": "Alk",
    "Calcium": "Ca",
    "Sodium": "Na",
    "Magnesium": "Mg",
    "pH": "pH",
    "TDS": "TDS"
}

for analyte, fpath in input_files.items():
    print(f"Processing INFO for {analyte}...")
    df = pd.read_csv(fpath, sep=",", nrows=5000)  # just sample a chunk, metadata is repeated

    # Keep unique site/param combos
    meta = df[[
        "site_no", "station_nm", "CharacteristicName", "unit",
        "DrainageAreaMeasure.MeasureValue", "DrainageAreaMeasure.MeasureUnitCode"
    ]].drop_duplicates("site_no")

    # Convert drainage area to km² if needed
    def convert_area(val, unit):
        if pd.isna(val):
            return pd.NA
        if isinstance(val, str):
            try:
                val = float(val)
            except ValueError:
                return pd.NA
        if unit.strip().lower() in ["sq mi", "square miles", "mi2"]:
            return val * 2.58999
        return val  # already km²
    meta["drainSqKm"] = meta.apply(
        lambda r: convert_area(r["DrainageAreaMeasure.MeasureValue"], r["DrainageAreaMeasure.MeasureUnitCode"]),
        axis=1
    )

    # Add abbreviations
    meta["constitAbbrev"] = meta["CharacteristicName"].map(constit_map)

    # Select and rename columns
    meta_out = meta.rename(columns={
        "unit": "param.units",
        "station_nm": "shortName",
        "CharacteristicName": "paramShortName"
    })[["param.units", "shortName", "paramShortName", "constitAbbrev", "drainSqKm"]]

    # Write per-site INFO files
    for site, site_df in meta_out.groupby(meta["site_no"]):
        out_file = info_out / f"{site}_{analyte}_INFO.csv"
        site_df.to_csv(out_file, index=False)


In [1]:
import pandas as pd
merged_with_q_with_omega = pd.read_csv(
    "/home/st929/project/Shuang_Project/merged_with_q_t_omega_na.csv"
)# 1. Sites that ever exceed 50
sites_exceed = merged_with_q_with_omega.loc[
    merged_with_q_with_omega["omega_calcite"] >= 0, "site_no"
].unique()

# 2. Sites with at least 100 measurements
site_counts = merged_with_q_with_omega["site_no"].value_counts()
sites_100plus = site_counts[site_counts >= 100].index

# 3. Intersection: sites that meet both criteria
sites_to_keep = set(sites_exceed).intersection(sites_100plus)

# 4. Filter dataframe
filtered_df = merged_with_q_with_omega[
    merged_with_q_with_omega["site_no"].isin(sites_to_keep)
]

print("Number of sites kept:", filtered_df["site_no"].nunique())
print("Total rows kept:", filtered_df.shape[0])

Number of sites kept: 885
Total rows kept: 164110


## end of this script. identified total of 885 sites that will be used in this study. further processing is in the next jupyter notebook